# 02 — LightGBM Baseline v0

**Hazırlayan:** Ömer Faruk Kara (Kaptan)  
**Tarih:** 3 Temmuz 2026  
**Görev:** İlk LightGBM baseline iskeletini kur, modelin az feature ile çalıştığını doğrula.

---

## Bu Notebook Ne Yapıyor?

1. Veriyi yükle ve birleştir (`src/data.py`)
2. Negatif örnekleme ile tam eğitim seti oluştur (`src/negative_sampling.py`)
3. Feature'ları hesapla (`src/features.py`)
4. LightGBM modeli eğit (5-Fold Stratified CV)
5. Macro-F1 skorunu ölç (`src/metrics.py`)
6. Feature importance çıkar

> **Not:** Bu bir baseline'dır. Amaç sistemi çalışır hale getirmek, mükemmel sonuç değil.

## 0. Kurulum ve İçe Aktarma

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Proje kök dizinini Python yoluna ekle
# Bu sayede 'src' klasöründeki modülleri içe aktarabiliyoruz
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data             import load_terms, load_items, merge_pairs
from src.features         import build_features, FEATURE_COLS
from src.negative_sampling import build_training_set
from src.metrics          import macro_f1_from_proba, find_best_threshold, get_stratified_kfold

# LightGBM — hızlı gradient boosting kütüphanesi
try:
    import lightgbm as lgb
    print('LightGBM versiyonu:', lgb.__version__)
except ImportError:
    print('LightGBM yuklu degil. Yukleniyor...')
    os.system('pip install lightgbm -q')
    import lightgbm as lgb

print('Tum kutuphaneler yuklendi.')

## 1. Veri Yükleme

In [ ]:
# Veri klasörü yolu
DATA_DIR = os.path.join(PROJECT_ROOT, 'datasets')

print('Veriler yukleniyor...')
terms_df = load_terms(os.path.join(DATA_DIR, 'terms.csv'))
items_df  = load_items(os.path.join(DATA_DIR, 'items.csv'))

# Ham eğitim çiftleri (sadece pozitif, label=1)
train_raw = pd.read_csv(
    os.path.join(DATA_DIR, 'training_pairs.csv'),
    dtype={'id': 'string', 'term_id': 'string', 'item_id': 'string', 'label': 'int8'}
)

print(f'Sorgular (terms): {len(terms_df):,}')
print(f'Urunler (items) : {len(items_df):,}')
print(f'Egitim ciftleri : {len(train_raw):,} (tumu pozitif)')

## 2. Negatif Örnekleme — Eğitim Seti Oluşturma

Baseline için **1000 pozitif** örnek alıp **3:1 oranında** negatif üretiyoruz.  
Bu hız için; tam eğitimde tüm 250K pozitif kullanılacak.

In [ ]:
# Baseline için küçük örnek (hız için)
# Tam eğitimde: SAMPLE_SIZE = None
SAMPLE_SIZE    = 5_000   # Kaç pozitif örnek kullanılacak
NEGATIVE_RATIO = 3       # Her pozitif için kaç negatif
RANDOM_SEED    = 42      # Takım standardı: her yerde 42

# Pozitiflerden örnek al
pos_sample = train_raw.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
print(f'{SAMPLE_SIZE:,} pozitif ornek alindi.')

# Negatif üret ve birleştir
full_train = build_training_set(
    pos_sample, items_df,
    ratio=NEGATIVE_RATIO,
    random_state=RANDOM_SEED,
    verbose=True
)

print(f'\nEgitim seti boyutu: {len(full_train):,}')
print(full_train['label'].value_counts().rename({0: 'Negatif (0)', 1: 'Pozitif (1)})}.to_string())

## 3. Veri Birleştirme (Merge) ve Feature Üretimi

In [ ]:
# Eğitim setine sorgu ve ürün bilgilerini birleştir
print('Merge yapiliyor...')
merged = full_train.merge(terms_df, on='term_id', how='left')
merged = merged.merge(items_df, on='item_id', how='left')
print(f'Merge tamamlandi. Boyut: {merged.shape}')

# Feature'ları hesapla
print('\nFeature hesaplaniyor...')
merged = build_features(merged)

# Sonucu kontrol et
print('\nFeature istatistikleri:')
print(merged[FEATURE_COLS].describe().round(3).to_string())

## 4. LightGBM Eğitimi — 5-Fold Stratified CV

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Özellik matrisi ve hedef etiket
X = merged[FEATURE_COLS]
y = merged['label']

print(f'X boyutu: {X.shape}  |  Pozitif oran: {y.mean():.1%}')

# LightGBM parametreleri (baseline için konservatif değerler)
lgbm_params = {
    'objective'        : 'binary',           # İkili sınıflandırma problemi
    'metric'           : 'binary_logloss',   # Eğitim sırasında bu metriği izle
    'learning_rate'    : 0.05,               # Öğrenme hızı (küçük = daha stabil)
    'num_leaves'       : 31,                 # Ağaçların yaprak sayısı (düşük = overfit az)
    'min_child_samples': 20,                 # Yaprakta minimum örnek (overfit önler)
    'subsample'        : 0.8,                # Her ağaçta verinin %80'ini kullan
    'colsample_bytree' : 0.8,                # Her ağaçta feature'ların %80'ini kullan
    'verbose'          : -1,                 # Gürültüsüz mod
    'random_state'     : 42,
}

# 5-Fold Stratified CV çalıştır
skf = get_stratified_kfold(n_splits=5, random_state=42)
fold_scores = []
oof_preds   = np.zeros(len(X))  # Out-of-fold tahminler (threshold optimizasyonu için)
trained_models = []

print('\n5-Fold Stratified CV basliyor...')
print('-' * 45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train_f, X_val_f = X.iloc[train_idx], X.iloc[val_idx]
    y_train_f, y_val_f = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM veri formatına dönüştür
    dtrain = lgb.Dataset(X_train_f, label=y_train_f)
    dval   = lgb.Dataset(X_val_f,   label=y_val_f)

    # Modeli eğit
    model = lgb.train(
        lgbm_params,
        dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(30, verbose=False),  # 30 round gelişme yoksa dur
            lgb.log_evaluation(period=-1),           # Log bastırma
        ]
    )
    trained_models.append(model)

    # Validation tahmini
    val_proba = model.predict(X_val_f)
    oof_preds[val_idx] = val_proba

    # Macro-F1 hesapla
    fold_f1 = macro_f1_from_proba(y_val_f, val_proba, threshold=0.5)
    fold_scores.append(fold_f1)
    print(f'  Fold {fold}/5  |  Macro-F1: {fold_f1:.4f}  |  Best iter: {model.best_iteration}')

print('-' * 45)
print(f'  ORT. Macro-F1: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})')

## 5. Threshold Optimizasyonu

Varsayılan 0.5 eşiği her zaman optimal değildir.  
Out-of-fold tahminler üzerinde en iyi eşiği arıyoruz.

In [ ]:
best_thresh, best_score, all_results = find_best_threshold(y.values, oof_preds)
print(f'Varsayilan (t=0.50) Macro-F1 : {macro_f1_from_proba(y, oof_preds, 0.5):.4f}')
print(f'En iyi    (t={best_thresh:.2f}) Macro-F1 : {best_score:.4f}')
print(f'\nFark: +{best_score - macro_f1_from_proba(y, oof_preds, 0.5):.4f}')

## 6. Feature Importance — Hangi Feature'lar Önemli?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Tüm fold'ların feature importance'larını ortala
importance_arr = np.zeros(len(FEATURE_COLS))
for model in trained_models:
    importance_arr += model.feature_importance(importance_type='gain')
importance_arr /= len(trained_models)

# Tablo olarak göster
feat_imp = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': importance_arr
}).sort_values('importance', ascending=False)

print('Feature Importance (gain):')  
print('-' * 40)
for _, row in feat_imp.iterrows():
    bar = '#' * int(row['importance'] / feat_imp['importance'].max() * 30)
    print(f"  {row['feature']:<28} {bar} ({row['importance']:.1f})")

## 7. Baseline Sonuç Özeti

In [ ]:
print('=' * 55)
print('  BASELINE v0 SONUC OZETI')
print('=' * 55)
print(f'  Model          : LightGBM')
print(f'  Egitim seti    : {SAMPLE_SIZE:,} pozitif + {SAMPLE_SIZE*NEGATIVE_RATIO:,} negatif')
print(f'  Negatif oran   : {NEGATIVE_RATIO}:1 (random)')
print(f'  Feature sayisi : {len(FEATURE_COLS)}')
print(f'  CV             : 5-Fold Stratified')
print(f'  Ort. Macro-F1  : {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')
print(f'  En iyi thresh  : {best_thresh}')
print(f'  Optimized F1   : {best_score:.4f}')
print('=' * 55)
print()
print('Sonraki adimlar:')
print('  - Tum 250K pozitif ile tam egitim (Gun 4)')
print('  - TF-IDF cosine feature ekleme (Muhammed - Gun 3/4)')
print('  - BM25 hard negative (Mustafa Mert - Gun 7)')
print('  - Ilk Kaggle submission (Omer Faruk - Gun 5)')